In [0]:
--SELECT * FROM demo_banking_bronze.banking_before_and_after_demo.qcdmbusinessaccounttransactions_raw;

--USE qdp_database; 
SELECT * FROM qdp_refcodes_all.tennant;




-- 01. Create variables for use in this SQL Script

-- 01-1. Create a tennant identifier for all data SQL Statements in this script
SET session my.vars.tennant = '999';


-- 01-2. Create a point in time run_timestamp for all data inserts
SET session my.vars.default_start_date = '01-01-1900';
SET session my.vars.default_end_date = '31-12-2999';

SELECT current_setting('my.vars.default_start_date')::text;
SELECT current_setting('my.vars.default_end_date')::text;
SELECT current_setting('my.vars.tennant')::int;


--02. Back out data from Tennant before populating with new data


--02-2. Remove all Transaction records for this Tennant
SET search_path TO qdp_transactions_all;
DELETE FROM sat_transaction s
 WHERE s.transaction_id IN (
  SELECT h.transaction_id 
    FROM hub_transaction h 
    WHERE h.tennant_id = current_setting('my.vars.tennant')::int)
;

DELETE FROM hub_transaction WHERE tennant_id = current_setting('my.vars.tennant')::int;


--02-3. Remove all Account records for this Tennant
SET search_path TO qdp_accounts_all;
DELETE FROM sat_account s
 WHERE s.account_id IN (
  SELECT h.account_id 
    FROM hub_account h 
    WHERE h.tennant_id = current_setting('my.vars.tennant')::int)
;

DELETE FROM hub_account WHERE tennant_id = current_setting('my.vars.tennant')::int;




--02-4. Remove all Individual Customer records for this Tennant
SET search_path TO qdp_individual_customers;
DELETE FROM sat_individual_customers s
 WHERE s.individual_customer_id IN (
  SELECT h.individual_customer_id
    FROM hub_individual_customers h 
    WHERE h.tennant_id = current_setting('my.vars.tennant')::int)
;

DELETE FROM hub_individual_customers WHERE tennant_id = current_setting('my.vars.tennant')::int;



--02-5. Remove all Individual records for this Tennant
SET search_path TO qdp_individuals_all;
DELETE FROM sat_human_name s
 WHERE s.record_source_id = current_setting('my.vars.tennant')::int
;

DELETE FROM sat_individual s
 WHERE s.individual_id IN (
  SELECT h.individual_id 
    FROM hub_individual h 
    WHERE h.tennant_id = current_setting('my.vars.tennant')::int)
;

DELETE FROM hub_individual WHERE tennant_id = current_setting('my.vars.tennant')::int;




--02-6. Remove all Organisation Customer records for this Tennant
SET search_path TO qdp_organisation_customers;
DELETE FROM sat_organisation_customers s
 WHERE s.organisation_customer_id IN (
  SELECT h.organisation_customer_id
    FROM hub_organisation_customers h 
    WHERE h.tennant_id = current_setting('my.vars.tennant')::int)
;

DELETE FROM hub_organisation_customers WHERE tennant_id = current_setting('my.vars.tennant')::int;


--02-7. Remove all Organisation records for this Tennant
SET search_path TO qdp_individual_customers;
DELETE FROM qdp_organisations_all.sat_organisation s
  WHERE s.record_source_id = 999
-- WHERE s.organisation_id IN (
--  SELECT h.organisation_id 
--    FROM demo_banking_silver.qdp_organisations_all.hub_organisation h 
--    WHERE h.tennant_id = current_setting('my.vars.tennant')::int)
;

DELETE FROM qdp_organisations_all.hub_organisation WHERE tennant_id = current_setting('my.vars.tennant')::int;



-- 03.  Create a working view for all relavent CDM AcccountTransaction records
SELECT * FROM demo_banking_bronze.banking_before_and_after_demo.qcdmbusinessaccounttransactions_raw;
/**
--SELECT * FROM demo_banking_bronze.banking_before_and_after_demo.qcdmbusinessaccounttransactions_raw;
--qdp_database.qdp_locations_all

SET search_path TO qdp_location_all;
CREATE OR REPLACE TEMP VIEW raw_accounts_view AS
SELECT accounts
FROM demo_banking_bronze.banking_before_and_after_demo.qcdmbusinessaccounttransactions_raw;


SELECT * FROM raw_accounts_view;


-- 04.  Get Accounts from CDM AcccountTransactions
CREATE OR REPLACE TEMP VIEW raw_accounts_flat_view AS
SELECT 
    accounts.account.identifiers.entityId,
    accounts.account.identifiers.accountNumber,
    accounts.account.identifiers.accountHolder,
    accounts.account.accountType.code AS accountTypeCode,
    accounts.account.accountType.rawCode AS accountTypeRawCode,
    accounts.account.status.code AS statusCode,
    accounts.account.status.rawCode AS statusRawCode,
    accounts.account.balance.currency.rawCode AS balanceCurrencyRawCode,
    accounts.account.countryCode.code AS countryCode,
    accounts.account.countryCode.rawCode AS countryRawCode,
    accounts.account.isBusinessAccount,
    accounts.account.isMultipleAcccountHolders,
    accounts.account.isPersonalAccount,
    accounts.account.parties,
    accounts.transactions
FROM (
  SELECT explode(accounts) AS accounts 
  FROM demo_banking_bronze.banking_before_and_after_demo.qcdmbusinessaccounttransactions_raw acc
) AS exploded_accounts
LIMIT 1000000;


SELECT * FROM raw_accounts_flat_view;


-- 05.  Get Account Parties from CDM Accounts
CREATE OR REPLACE TEMP VIEW raw_account_parties_view AS
SELECT 
    accountNumber,
    party.*
 FROM (
  SELECT accview.accountNumber AS accountNumber, explode(accview.parties) AS party 
  FROM raw_accounts_flat_view accview
) AS exploded_account_parties
LIMIT 1000000;

SELECT * FROM raw_account_parties_view;



CREATE OR REPLACE TEMP VIEW raw_account_parties_flat_view AS
SELECT 
    accountNumber,
    party.entityId AS partyEntityId,
    party.name AS partyName,
    party.partyRole.code AS roleCode,
    party.isIndividual,
    party.isOrganisation,
    party.individual.identifiers.entityId AS individualEntityId,
    party.individual,
    party.organisation.identifiers.entityId AS organisationEntityId,
    party.organisation,
    party.address.identifiers.entityId AS addressEntityId,
    party.address
 FROM raw_account_parties_view
LIMIT 1000000;


SELECT * 
  FROM raw_account_parties_flat_view;



-- 06.  Get Account Transactions from CDM Accounts
CREATE OR REPLACE TEMP VIEW raw_account_transactions_view AS
SELECT 
    uuid() as transactionId,
    accountNumber,
    transaction.*
 FROM (
  SELECT accview.accountNumber AS accountNumber, explode(accview.transactions) AS transaction 
  FROM raw_accounts_flat_view accview
) AS exploded_account_transactionss
LIMIT 1000000;


SELECT * FROM raw_account_transactions_view;



CREATE OR REPLACE TEMP VIEW raw_account_transactions_flat_view AS
SELECT 
    transactionId,
    accountNumber,
    accountTransaction.amount.raw AS amountRaw,
    accountTransaction.amount.currency.rawCode AS amountCurrencyRawCode,
    accountTransaction.beneficaryCountryCode.rawCode AS beneficiaryCountryRawCode,
    accountTransaction.beneficiaryAccountNumber AS beneficiaryAccountNumber,
    accountTransaction.beneficiaryAmount.currency.rawCode AS beneficiaryAmountCurrencyRawCode,
    accountTransaction.creditOrDebit.rawCode AS creditOrDebitRawCode,
    accountTransaction.identifiers.sourceDataLineage.itSystem AS sourceDataLineageItSystem,
    accountTransaction.originatorAccountNumber AS originatorAccountNumber,
    accountTransaction.originatorAmount.currency.rawCode AS originatorAmountCurrencyRawCode,
    accountTransaction.originatorAmount.raw AS originatorAmountRaw,
    accountTransaction.originatorCountryCode.rawCode AS originatorCountryRawCode,
    accountTransaction.originatorName AS originatorName,
    accountTransaction.postingDateTime.raw AS postingDateTimeRaw,
    accountTransaction.postingDateTime.rawFormat AS postingDateTimeRawFormat,
    accountTransaction.transactionCategory.code AS transactionCategoryCode
 FROM raw_account_transactions_view
LIMIT 1000000;



SELECT * FROM raw_account_transactions_flat_view;



-- 10.  Create Address Records contained with the Account Transactions CDM JSON Document

-- 10.1 Insert into hub_address
INSERT INTO demo_banking_silver.qdp_locations_all.hub_address
(address_entity_id, 
 tennant_id)
 SELECT ap.addressEntityId, 
       :p_tennant_id
FROM raw_account_parties_flat_view ap
WHERE ap.address is NOT NULL;

SELECT * FROM demo_banking_silver.qdp_locations_all.hub_address where tennant_id = 999;



-- 10.2 Insert into sat_address
INSERT INTO demo_banking_silver.qdp_locations_all.sat_address
    (sat_address_id,
     address_id, 
     load_datetime, 
     record_source_id, 
     postal_code,
     city,
     country,
     country_raw_code
    )
SELECT
  monotonically_increasing_id() AS sat_address_id,
  h.address_id AS address_id,
  run_timestamp AS load_datetime,
  try_cast(999 AS BIGINT) AS record_source_id,
  ap.address.postalCode AS postal_code,
  ap.address.city AS city,
  ap.address.country AS country,
  ap.address.countryCode.rawCode AS country_raw_code

FROM raw_account_parties_flat_view ap
JOIN demo_banking_silver.qdp_locations_all.hub_address h
  ON h.address_entity_id = ap.addressEntityId
WHERE ap.address is NOT NULL;

  
SELECT * FROM demo_banking_silver.qdp_locations_all.sat_address where record_source_id = 999;



-- 11.  Create Organisation Records contained with the Account Transactions CDM JSON Document
CREATE OR REPLACE TEMP VIEW raw_organisation_customers_view AS
SELECT DISTINCT partyName, 
                roleCode,
                isOrganisation,
                organisationEntityId, 
                organisation.name, 
                organisation.identifiers.companyRegistrationNumber.value AS companyRegistrationNumber, 
                organisation.identifiers.sourceDataLineage.class AS sourceDataLineageClass 
  FROM raw_account_parties_flat_view 
  WHERE 
    roleCode = "Account Holder" AND 
    isOrganisation = true
;

SELECT * FROM raw_organisation_customers_view;

-- 11.1 Insert into hub_organisation
INSERT INTO demo_banking_silver.qdp_organisations_all.hub_organisation
(organisation_entity_id, 
 tennant_id)
 SELECT oc.organisationEntityId, 
       :p_tennant_id
FROM raw_organisation_customers_view oc
WHERE oc.organisationEntityId is NOT NULL;

SELECT * FROM demo_banking_silver.qdp_organisations_all.hub_organisation where tennant_id = 999;


-- 11.2 Insert into sat_organisation
INSERT INTO demo_banking_silver.qdp_organisations_all.sat_organisation
    (
--     sat_organisation_id,
     organisation_id, 
     load_datetime, 
     record_source_id, 
     organisation_name,
     organisation_level,
     primary_adddress_id,
     parent_organisation_id,
     company_registration_number,
     data_source_raw_code
    )
SELECT
--  monotonically_increasing_id() AS sat_organisation_id,
  h.organisation_id AS organisation_id,
  run_timestamp AS load_datetime,
  try_cast(999 AS BIGINT) AS record_source_id,
  oc.name AS organisation_name,
  0,
  0,
  0,
  oc.companyRegistrationNumber AS company_registration_number,
  oc.sourceDataLineageClass AS data_source_raw_code

FROM raw_organisation_customers_view oc
JOIN demo_banking_silver.qdp_organisations_all.hub_organisation h
  ON h.organisation_entity_id = oc.organisationEntityId
WHERE oc.organisationEntityId is NOT NULL;


  
SELECT * FROM demo_banking_silver.qdp_organisations_all.sat_organisation where record_source_id = 999;






-- 12.  Create Individual Records contained with the Account Transactions CDM JSON Document

-- 12.1 Insert into hub_individual
INSERT INTO demo_banking_silver.qdp_individuals_all.hub_individual
(individual_entity_id, 
 tennant_id)
 SELECT ap.individualEntityId, 
       :p_tennant_id
FROM raw_account_parties_flat_view ap
WHERE ap.individual is NOT NULL;

SELECT * FROM demo_banking_silver.qdp_individuals_all.hub_individual where tennant_id = 999;


-- 12.2 Insert into sat_individual
INSERT INTO demo_banking_silver.qdp_individuals_all.sat_individual
    (individual_id, 
     load_datetime, 
     record_source_id,
     is_deceased,
     birth_date)
SELECT
  h.individual_id AS individual_id,
  run_timestamp AS load_datetime,
  try_cast(999 AS BIGINT) AS record_source_id,
  ap.individual.deceased.isDeceased AS is_deceased,
  TRY_TO_DATE(ap.individual.birthDate.raw, "yyyy-MM-dd") AS birth_date

FROM raw_account_parties_flat_view ap
JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
  ON h.individual_entity_id = ap.individualEntityId
WHERE ap.individual is NOT NULL;

SELECT * FROM demo_banking_silver.qdp_individuals_all.sat_individual where record_source_id = 999;



-- 12.3 Explode Party Individual Human Names

CREATE OR REPLACE TEMP VIEW raw_account_individual_names_view AS
SELECT 
    accountNumber,
    individualEntityId,
    humanName.*
 FROM (
  SELECT apin.accountNumber AS accountNumber, apin.individualEntityId AS individualEntityId, explode(apin.individual.humanNames) AS humanName 
  FROM raw_account_parties_flat_view apin
) AS exploded_account_individual_names
LIMIT 1000000;

SELECT * FROM raw_account_individual_names_view;

-- 12.4 Insert into sat_human_name
INSERT INTO demo_banking_silver.qdp_individuals_all.sat_human_name
    (individual_id, 
     load_datetime, 
     record_source_id,
     full_name,
     family,
     given1
--     given2
     )
SELECT
  h.individual_id AS individual_id,
  run_timestamp AS load_datetime,
  try_cast(999 AS BIGINT) AS record_source_id,
  ap.humanName.fullName AS full_name,
  ap.humanName.familyName AS family,
  ap.humanName.givenNames[0].givenName.given AS given1
  --  ap.humanName.givenNames[1].givenName.given) AS given2

FROM raw_account_individual_names_view ap
JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
  ON h.individual_entity_id = ap.individualEntityId
WHERE ap.humanName is NOT NULL;

SELECT * FROM demo_banking_silver.qdp_individuals_all.sat_human_name where record_source_id = 999;



-- 13.  Create Account Records contained with the Account Transactions CDM JSON Document
SELECT * FROM raw_accounts_flat_view acc;



-- 14.1 Insert into hub_acccount
INSERT INTO demo_banking_silver.qdp_accounts_all.hub_account
(tennant_id,
 account_entity_id 
 )
 SELECT :p_tennant_id,
        acc.entityId       
FROM raw_accounts_flat_view acc
;

SELECT * FROM demo_banking_silver.qdp_accounts_all.hub_account;



-- 14.2 Insert into sat_account
INSERT INTO demo_banking_silver.qdp_accounts_all.sat_account
    (account_id, 
     load_datetime, 
     record_source_id,
     account_number,
     account_name,
     type_raw_code,
     status_raw_code,
     country_raw_code,
     currency_raw_code,
     is_business_account
)
SELECT
  h.account_id AS account_id,
  current_timestamp() AS load_datetime,
  try_cast(999 AS BIGINT) AS record_source_id,
  acc.accountNumber AS account_number,
  acc.accountHolder AS account_name,
  acc.accountTypeRawCode AS type_raw_code,
  acc.statusRawCode AS status_raw_code,
  acc.countryRawCode AS country_raw_code,
  acc.balanceCurrencyRawCode AS currency_raw_code,
  acc.isBusinessAccount AS is_business_account

FROM raw_accounts_flat_view acc
JOIN demo_banking_silver.qdp_accounts_all.hub_account h
  ON h.account_entity_id = acc.entityId
;  

SELECT * FROM demo_banking_silver.qdp_accounts_all.sat_account WHERE record_source_id = 999;


-- 15.  Create Aggregated Transaction Records contained with the Account Transactions CDM JSON Document

CREATE OR REPLACE TEMP VIEW raw_accounts_flat_with_accountid_view AS
SELECT 
    h.account_id AS accountId,
    acc.*
 FROM raw_accounts_flat_view acc
JOIN demo_banking_silver.qdp_accounts_all.hub_account h
  ON h.account_entity_id = acc.entityId
LIMIT 1000000;



SELECT * FROM  raw_accounts_flat_with_accountid_view;


-- 15.1.  Get Account Transactions from CDM Accounts
CREATE OR REPLACE TEMP VIEW raw_account_transactions_view AS
SELECT 
    uuid() as transactionId,
    entityId AS accountEntityId,
    accountId,
    accountNumber,
    accountHolder,
    accountTypeRawCode,
    statusRawCode,
    countryRawCode,
    balanceCurrencyRawCode,
    transaction.*
 FROM (
  SELECT accview.entityId as entityId,
         accview.accountId AS accountId,
         accview.accountNumber AS accountNumber, 
         accview.accountHolder AS accountHolder, 
         accview.accountTypeRawCode AS accountTypeRawCode,
         accview.statusRawCode AS statusRawCode,
         accview.countryRawCode AS countryRawCode,
         accview.balanceCurrencyRawCode AS balanceCurrencyRawCode,
         explode(accview.transactions) AS transaction 
  FROM raw_accounts_flat_with_accountid_view accview
) AS exploded_account_transactionss
LIMIT 1000000;


SELECT * FROM raw_account_transactions_view;

SELECT * FROM raw_accounts_flat_view;



-- 15.2.  Flatten the individual Transaction records
CREATE OR REPLACE TEMP VIEW raw_account_transactions_flat_view AS
SELECT 
    transactionId,
    accountEntityId,
    accountId,
    accountNumber,
    accountHolder,
    accountTypeRawCode,
    statusRawCode,
    countryRawCode,
    balanceCurrencyRawCode,
    accountTransaction.identifiers.entityId AS transactionEntityId,
    accountTransaction.amount.raw AS amountRaw,
    accountTransaction.amount.currency.rawCode AS amountCurrencyRawCode,
    accountTransaction.beneficaryCountryCode.rawCode AS beneficiaryCountryRawCode,
    accountTransaction.beneficiaryAccountNumber AS beneficiaryAccountNumber,
    accountTransaction.beneficiaryAccountEntityId AS beneficiaryAccountEntityId,
    accountTransaction.beneficiaryname AS beneficiaryName,
    accountTransaction.beneficiaryEntityId AS beneficiaryEntityId,
    accountTransaction.beneficiaryAmount.currency.rawCode AS beneficiaryAmountCurrencyRawCode,
    accountTransaction.creditOrDebit.rawCode AS creditOrDebitRawCode,
    accountTransaction.identifiers.sourceDataLineage.itSystem AS sourceDataLineageItSystem,
    accountTransaction.originatorAccountNumber AS originatorAccountNumber,
    accountTransaction.originatorAccountEntityId AS originatorAccountEntityId,
    accountTransaction.originatorName AS originatorName,
    accountTransaction.originatorEntityId AS originatorEntityId,
    accountTransaction.originatorAmount.currency.rawCode AS originatorAmountCurrencyRawCode,
    accountTransaction.originatorAmount.raw AS originatorAmountRaw,
    accountTransaction.originatorCountryCode.rawCode AS originatorCountryRawCode,
    accountTransaction.postingDateTime.raw AS postingDateTimeRaw,
    accountTransaction.postingDateTime.rawFormat AS postingDateTimeRawFormat,
    accountTransaction.transactionCategory.code AS transactionCategoryCode
 FROM raw_account_transactions_view
LIMIT 1000000;

SELECT acct.* FROM raw_account_transactions_flat_view acct;

-- 15.3. Assign the correct bene_account_id or leave as defaulted zero value if does not exist
CREATE OR REPLACE TEMP VIEW raw_account_transactions_flat_baccid_view AS
SELECT 
    acctf.*,
    COALESCE(acc.account_id, 0) AS beneficiaryAccountId
FROM raw_account_transactions_flat_view acctf
LEFT JOIN demo_banking_silver.qdp_accounts_all.hub_account acc
ON acc.account_entity_id = acctf.beneficiaryAccountEntityId
;

SELECT * FROM raw_account_transactions_flat_baccid_view;



-- 15.4. Assign the correct orig_account_id or leave as defaulted zero value if does not exist
CREATE OR REPLACE TEMP VIEW raw_account_transactions_flat_baccid_oaccid_view AS
SELECT 
    acctfb.*,
    COALESCE(acc.account_id, 0) AS originatorAccountId
FROM raw_account_transactions_flat_baccid_view acctfb
LEFT JOIN demo_banking_silver.qdp_accounts_all.hub_account acc
ON acc.account_entity_id = acctfb.originatorAccountEntityId
;

SELECT * FROM raw_account_transactions_flat_baccid_oaccid_view;




-- 15.5 Insert into hub_aggregated_transactions
INSERT INTO demo_banking_silver.qdp_transactions_all.hub_transaction
(transaction_entity_id, 
 tennant_id,
 bene_account_entity_id,
 orig_account_entity_id,
 bene_account_id,
 orig_account_id,
 transaction_id_raw)
 SELECT acct.transactionEntityId, 
       :p_tennant_id,
       acct.beneficiaryAccountEntityId,
       acct.originatorAccountEntityId,
       acct.beneficiaryAccountId,
       acct.originatorAccountId,
       acct.transactionId
FROM raw_account_transactions_flat_baccid_oaccid_view acct
;

SELECT * FROM demo_banking_silver.qdp_transactions_all.hub_transaction where tennant_id = 999;



-- 15.6 Insert into sat_aggregated_transactions
INSERT INTO demo_banking_silver.qdp_transactions_all.sat_transaction
    (transaction_id, 
     load_datetime, 
     record_source_id,
     account_number,
     payment_method_raw_code,
     data_source_raw_code,
     debit_or_credit_raw_code,
     amount,
     datetime,
     original_country_raw_code,
     country_code,
     original_account_number,
     original_amount,
     original_currency_raw_code,
     currency_code
    )
SELECT
  h.transaction_id AS transaction_id,
  run_timestamp AS load_datetime,
  try_cast(999 AS BIGINT) AS record_source_id,
  acct.accountNumber,
  acct.transactionCategoryCode AS type_code,
  acct.sourceDataLineageItSystem,
  acct.creditOrDebitRawCode,
  acct.amountRaw,
  TRY_TO_DATE(acct.postingDateTimeRaw, "yyyy MM dd") AS transaction_date_time,
  acct.originatorCountryRawCode,
  acct.beneficiaryCountryRawCode,
  acct.originatorAccountNumber,
  acct.originatorAmountRaw,
  acct.originatorAmountCurrencyRawCode,
  acct.beneficiaryAmountCurrencyRawCode

FROM raw_account_transactions_flat_baccid_oaccid_view acct
JOIN demo_banking_silver.qdp_transactions_all.hub_transaction h
  ON h.transaction_entity_id = acct.transactionEntityId
;  


SELECT * FROM demo_banking_silver.qdp_transactions_all.sat_transaction where record_source_id = 999;







**/